#  Preprocesamiento de Texto para Detección de Toxicidad

**Autor:** Gema
**Fecha:** 2026  
**Objetivo:** Limpiar y preparar el texto del dataset para entrenamiento del modelo

---

##  Qué vamos a hacer en este notebook

1. ✅ Cargar el dataset original
2. ✅ Limpiar el texto (minúsculas, URLs, caracteres especiales)
3. ✅ Eliminar stopwords en inglés
4. ✅ Aplicar stemming o lematización
5. ✅ Guardar dataset limpio con la columna `text_procesado`

---

## 📊 Dataset

- **Entrada:** `data/raw/youtoxic_orginal.csv`
- **Salida:** `data/processed/youtoxic_comments_cleaned.csv`
- **Target:** IsToxic (binario: True/False)
- **Balance:** 53.8% Normal vs 46.2% Tóxico

---

## 🛠️ Técnicas de Preprocesamiento

1. **Convertir a minúsculas:** "HELLO" → "hello"
2. **Eliminar URLs:** "Check https://example.com" → "Check"
3. **Eliminar menciones:** "@user text" → "text"
4. **Eliminar números:** "I hate 123" → "I hate"
5. **Eliminar caracteres especiales:** "What!!!" → "What"
6. **Eliminar stopwords:** "this is a test" → "test"
7. **Stemming:** "running runs ran" → "run run run"

---

**Comenzamos →**

In [3]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import re
import warnings

# Librerías de NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

# Descargar recursos de NLTK (solo primera vez)
print(" Descargando recursos de NLTK...")
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print(" Librerías importadas correctamente")
print(f" Stopwords en inglés: {len(stopwords.words('english'))} palabras")

 Descargando recursos de NLTK...
 Librerías importadas correctamente
 Stopwords en inglés: 198 palabras


In [6]:
# Cargar el dataset original
print("="*80)
print("📂 CARGANDO DATASET")
print("="*80)

# Cargar
df = pd.read_csv('../data/raw/youtoxic_original.csv')

print(f" Dataset cargado correctamente")
print(f"\n Dimensiones: {df.shape[0]:,} comentarios × {df.shape[1]} columnas")
print(f"\n Columnas disponibles:")
print(df.columns.tolist())

# Mostrar primeras 3 filas
print(f"\n{'='*80}")
print(" PRIMERAS 3 FILAS")
print(f"{'='*80}")
print(df[['Text', 'IsToxic']].head(5))

📂 CARGANDO DATASET
 Dataset cargado correctamente

 Dimensiones: 1,000 comentarios × 15 columnas

 Columnas disponibles:
['CommentId', 'VideoId', 'Text', 'IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

 PRIMERAS 3 FILAS
                                                                                                                                                                                                      Text  \
0  If only people would just take a step back and not make this case about them, because it wasn't about anyone except the two people in that situation.  To lump yourself into this mess and take matt...   
1                                                               Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.   
2  \r\nDont you reckon them 'black 

In [7]:
# Seleccionar solo las columnas necesarias
print("="*80)
print(" SELECCIONANDO COLUMNAS NECESARIAS")
print("="*80)

# Columnas que vamos a mantener
columnas_necesarias = ['CommentId', 'VideoId', 'Text', 'IsToxic']

# Filtrar DataFrame
df = df[columnas_necesarias]

print(f" Columnas seleccionadas: {df.columns.tolist()}")
print(f"\n Nuevo tamaño: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Verificar balance de IsToxic
print(f"\n⚖️ BALANCE DEL TARGET:")
print(df['IsToxic'].value_counts())
print(f"\nPorcentajes:")
print(df['IsToxic'].value_counts(normalize=True) * 100)

 SELECCIONANDO COLUMNAS NECESARIAS
 Columnas seleccionadas: ['CommentId', 'VideoId', 'Text', 'IsToxic']

 Nuevo tamaño: 1,000 filas × 4 columnas

⚖️ BALANCE DEL TARGET:
IsToxic
False    538
True     462
Name: count, dtype: int64

Porcentajes:
IsToxic
False    53.8
True     46.2
Name: proportion, dtype: float64


In [8]:
# Funciones de preprocesamiento de texto

def convertir_minusculas(texto):
    """Convierte el texto a minúsculas"""
    return texto.lower()

def eliminar_urls(texto):
    """Elimina URLs del texto"""
    # Patrón regex para URLs
    patron_url = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    texto = re.sub(patron_url, '', texto)
    return texto

def eliminar_menciones(texto):
    """Elimina menciones de usuarios (@usuario)"""
    texto = re.sub(r'@\w+', '', texto)
    return texto

def eliminar_numeros(texto):
    """Elimina números del texto"""
    texto = re.sub(r'\d+', '', texto)
    return texto

def eliminar_caracteres_especiales(texto):
    """Elimina caracteres especiales, solo deja letras y espacios"""
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)
    return texto

def eliminar_espacios_extra(texto):
    """Elimina espacios múltiples"""
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def eliminar_stopwords(texto):
    """Elimina stopwords en inglés"""
    stop_words = set(stopwords.words('english'))
    palabras = texto.split()
    palabras_filtradas = [palabra for palabra in palabras if palabra not in stop_words]
    return ' '.join(palabras_filtradas)

def aplicar_lematizacion(texto):
    """Aplica lematización a cada palabra"""
    lemmatizer = WordNetLemmatizer()
    palabras = texto.split()
    palabras_lematizadas = [lemmatizer.lemmatize(palabra) for palabra in palabras]
    return ' '.join(palabras_lematizadas)

print("✅ Funciones de preprocesamiento creadas correctamente")
print("\nFunciones disponibles:")
print("  1. convertir_minusculas()")
print("  2. eliminar_urls()")
print("  3. eliminar_menciones()")
print("  4. eliminar_numeros()")
print("  5. eliminar_caracteres_especiales()")
print("  6. eliminar_espacios_extra()")
print("  7. eliminar_stopwords()")
print("  8. aplicar_lematizacion()")

✅ Funciones de preprocesamiento creadas correctamente

Funciones disponibles:
  1. convertir_minusculas()
  2. eliminar_urls()
  3. eliminar_menciones()
  4. eliminar_numeros()
  5. eliminar_caracteres_especiales()
  6. eliminar_espacios_extra()
  7. eliminar_stopwords()
  8. aplicar_lematizacion()


 # Vamos a probar que funcionan correctamente con UN comentario de ejemplo.

In [10]:
# Probar las funciones con un ejemplo real
print("="*80)
print(" PRUEBA DE FUNCIONES DE LIMPIEZA")
print("="*80)

# Tomar un comentario real del dataset
comentario_ejemplo = df['Text'][1]  # El segundo comentario

print(f"\n📝 COMENTARIO ORIGINAL:")
print(f"'{comentario_ejemplo}'")
print(f"\nLongitud: {len(comentario_ejemplo)} caracteres")

# Aplicar cada función paso a paso
print(f"\n{'='*80}")
print("APLICANDO CADA PASO:")
print(f"{'='*80}")

# Paso 1: Minúsculas
texto = convertir_minusculas(comentario_ejemplo)
print(f"\n1 Minúsculas:")
print(f"'{texto}'")

# Paso 2: Eliminar URLs
texto = eliminar_urls(texto)
print(f"\n Sin URLs:")
print(f"'{texto}'")

# Paso 3: Eliminar menciones
texto = eliminar_menciones(texto)
print(f"\n Sin menciones:")
print(f"'{texto}'")

# Paso 4: Eliminar números
texto = eliminar_numeros(texto)
print(f"\n Sin números:")
print(f"'{texto}'")

# Paso 5: Eliminar caracteres especiales
texto = eliminar_caracteres_especiales(texto)
print(f"\n Sin caracteres especiales:")
print(f"'{texto}'")

# Paso 6: Eliminar espacios extra
texto = eliminar_espacios_extra(texto)
print(f"\n Sin espacios extra:")
print(f"'{texto}'")

# Paso 7: Eliminar stopwords
texto = eliminar_stopwords(texto)
print(f"\n Sin stopwords:")
print(f"'{texto}'")

# Paso 8: Lematización
texto = aplicar_lematizacion(texto)
print(f"\n Lematizado:")
print(f"'{texto}'")

print(f"\n{'='*80}")
print(f" RESULTADO FINAL:")
print(f"{'='*80}")
print(f"Original: '{comentario_ejemplo}'")
print(f"Procesado: '{texto}'")
print(f"\nLongitud original: {len(comentario_ejemplo)} caracteres")
print(f"Longitud procesado: {len(texto)} caracteres")
print(f"Reducción: {len(comentario_ejemplo) - len(texto)} caracteres ({(1 - len(texto)/len(comentario_ejemplo))*100:.1f}%)")

 PRUEBA DE FUNCIONES DE LIMPIEZA

📝 COMENTARIO ORIGINAL:
'Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.'

Longitud: 138 caracteres

APLICANDO CADA PASO:

1 Minúsculas:
'law enforcement is not trained to shoot to apprehend.  they are trained to shoot to kill.  and i thank wilson for killing that punk bitch.'

 Sin URLs:
'law enforcement is not trained to shoot to apprehend.  they are trained to shoot to kill.  and i thank wilson for killing that punk bitch.'

 Sin menciones:
'law enforcement is not trained to shoot to apprehend.  they are trained to shoot to kill.  and i thank wilson for killing that punk bitch.'

 Sin números:
'law enforcement is not trained to shoot to apprehend.  they are trained to shoot to kill.  and i thank wilson for killing that punk bitch.'

 Sin caracteres especiales:
'law enforcement is not trained to shoot to apprehend  they are trained to shoot to kill  and i thank 

In [11]:
# Función completa que aplica TODOS los pasos
def preprocesar_texto_completo(texto):
    """
    Aplica TODOS los pasos de limpieza en orden:
    1. Minúsculas
    2. Eliminar URLs
    3. Eliminar menciones
    4. Eliminar números
    5. Eliminar caracteres especiales
    6. Eliminar espacios extra
    7. Eliminar stopwords
    8. Lematización
    
    Args:
        texto (str): Texto original a limpiar
        
    Returns:
        str: Texto limpio y procesado
    """
    # Aplicar cada paso en orden
    texto = convertir_minusculas(texto)
    texto = eliminar_urls(texto)
    texto = eliminar_menciones(texto)
    texto = eliminar_numeros(texto)
    texto = eliminar_caracteres_especiales(texto)
    texto = eliminar_espacios_extra(texto)
    texto = eliminar_stopwords(texto)
    texto = aplicar_lematizacion(texto)
    
    return texto

print(" Función completa de preprocesamiento creada: preprocesar_texto_completo()")

# Probar la función completa con el mismo ejemplo
print("\n" + "="*80)
print(" PRUEBA DE LA FUNCIÓN COMPLETA")
print("="*80)

comentario_test = df['Text'][1]
comentario_limpio = preprocesar_texto_completo(comentario_test)

print(f"\nOriginal:")
print(f"'{comentario_test}'")
print(f"\nProcesado:")
print(f"'{comentario_limpio}'")

 Función completa de preprocesamiento creada: preprocesar_texto_completo()

 PRUEBA DE LA FUNCIÓN COMPLETA

Original:
'Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.'

Procesado:
'law enforcement trained shoot apprehend trained shoot kill thank wilson killing punk bitch'


In [12]:
# Aplicar preprocesamiento a TODO el dataset
print("="*80)
print(" APLICANDO PREPROCESAMIENTO A TODO EL DATASET")
print("="*80)

import time

print(f"\n Procesando {len(df):,} comentarios...")
print("Esto puede tardar un momento...\n")

# Iniciar timer
inicio = time.time()

# Aplicar la función a cada fila
df['text_procesado'] = df['Text'].apply(preprocesar_texto_completo)

# Calcular tiempo
tiempo_total = time.time() - inicio

print(f" Preprocesamiento completado")
print(f"⏱ Tiempo total: {tiempo_total:.2f} segundos")
print(f" Promedio: {tiempo_total/len(df)*1000:.2f} ms por comentario")

# Mostrar algunos ejemplos
print(f"\n{'='*80}")
print(" EJEMPLOS DE COMENTARIOS PROCESADOS")
print(f"{'='*80}")

for i in [0, 1, 2]:
    print(f"\n--- Ejemplo {i+1} ---")
    print(f"Original:  '{df['Text'].iloc[i][:100]}...'")
    print(f"Procesado: '{df['text_procesado'].iloc[i]}'")
    print(f"IsToxic:   {df['IsToxic'].iloc[i]}")

 APLICANDO PREPROCESAMIENTO A TODO EL DATASET

 Procesando 1,000 comentarios...
Esto puede tardar un momento...

 Preprocesamiento completado
⏱ Tiempo total: 1.31 segundos
 Promedio: 1.31 ms por comentario

 EJEMPLOS DE COMENTARIOS PROCESADOS

--- Ejemplo 1 ---
Original:  'If only people would just take a step back and not make this case about them, because it wasn't abou...'
Procesado: 'people would take step back make case wasnt anyone except two people situation lump mess take matter hand make kind protest selfish without rational thought investigation guy video heavily emotional hyped want heard get heard press never reasonable discussion kudos smerconish keeping level whole time letting masri make fool dare tore city protest make dishonor entire incident hate way since police brutality become epidemic wish everyone would stop pretending like knew exactly going there measurable amount people honestly witnessed incident none u clue way whole issue swung grand jury informed trust maj

In [13]:
# Verificar el preprocesamiento
print("="*80)
print(" VERIFICACIÓN DEL PREPROCESAMIENTO")
print("="*80)

# 1. Verificar que no hay valores vacíos
print("\n Verificar valores vacíos:")
print(f"   Textos originales vacíos: {df['Text'].isna().sum()}")
print(f"   Textos procesados vacíos: {df['text_procesado'].isna().sum()}")

# 2. Verificar textos que quedaron completamente vacíos (solo espacios)
textos_vacios = df[df['text_procesado'].str.strip() == '']
print(f"\n Textos que quedaron completamente vacíos: {len(textos_vacios)}")

if len(textos_vacios) > 0:
    print(f"\n   Ejemplos de textos que quedaron vacíos:")
    for i in range(min(3, len(textos_vacios))):
        idx = textos_vacios.index[i]
        print(f"\n   Original:  '{df.loc[idx, 'Text']}'")
        print(f"   Procesado: '{df.loc[idx, 'text_procesado']}'")
        print(f"   IsToxic:   {df.loc[idx, 'IsToxic']}")

# 3. Estadísticas de longitud
print(f"\n Estadísticas de longitud:")
print(f"\n   ANTES del preprocesamiento:")
print(f"   - Longitud promedio: {df['Text'].str.len().mean():.1f} caracteres")
print(f"   - Longitud mín-máx: {df['Text'].str.len().min()} - {df['Text'].str.len().max()}")

print(f"\n   DESPUÉS del preprocesamiento:")
print(f"   - Longitud promedio: {df['text_procesado'].str.len().mean():.1f} caracteres")
print(f"   - Longitud mín-máx: {df['text_procesado'].str.len().min()} - {df['text_procesado'].str.len().max()}")

reduccion = (1 - df['text_procesado'].str.len().mean() / df['Text'].str.len().mean()) * 100
print(f"\n    Reducción promedio: {reduccion:.1f}%")

# 4. Ver estructura final del DataFrame
print(f"\n Estructura final del DataFrame:")
print(f"\n{df.info()}")

# 5. Mostrar primeras 5 filas
print(f"\n Primeras 5 filas del dataset procesado:")
print(df[['Text', 'text_procesado', 'IsToxic']].head())

 VERIFICACIÓN DEL PREPROCESAMIENTO

 Verificar valores vacíos:
   Textos originales vacíos: 0
   Textos procesados vacíos: 0

 Textos que quedaron completamente vacíos: 0

 Estadísticas de longitud:

   ANTES del preprocesamiento:
   - Longitud promedio: 186.0 caracteres
   - Longitud mín-máx: 3 - 4474

   DESPUÉS del preprocesamiento:
   - Longitud promedio: 112.6 caracteres
   - Longitud mín-máx: 3 - 2558

    Reducción promedio: 39.4%

 Estructura final del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CommentId       1000 non-null   object
 1   VideoId         1000 non-null   object
 2   Text            1000 non-null   object
 3   IsToxic         1000 non-null   bool  
 4   text_procesado  1000 non-null   object
dtypes: bool(1), object(4)
memory usage: 32.4+ KB

None

 Primeras 5 filas del dataset procesado:
             

In [14]:
# Guardar el dataset procesado
print("="*80)
print(" GUARDANDO DATASET PROCESADO")
print("="*80)

# Ruta de salida
ruta_salida = '../data/processed/youtoxic_comments_cleaned.csv'

# Guardar
df.to_csv(ruta_salida, index=False)

print(f" Dataset guardado exitosamente en: {ruta_salida}")
print(f"\ Información del archivo guardado:")
print(f"   - Filas: {len(df):,}")
print(f"   - Columnas: {len(df.columns)}")
print(f"   - Columnas guardadas: {df.columns.tolist()}")

# Verificar que se guardó correctamente
import os
if os.path.exists(ruta_salida):
    tamaño = os.path.getsize(ruta_salida) / 1024  # KB
    print(f"   - Tamaño del archivo: {tamaño:.1f} KB")
    print(f"\n Archivo verificado y guardado correctamente")
else:
    print(f"\n❌ ERROR: El archivo no se guardó correctamente")

 GUARDANDO DATASET PROCESADO
 Dataset guardado exitosamente en: ../data/processed/youtoxic_comments_cleaned.csv
\ Información del archivo guardado:
   - Filas: 1,000
   - Columnas: 5
   - Columnas guardadas: ['CommentId', 'VideoId', 'Text', 'IsToxic', 'text_procesado']
   - Tamaño del archivo: 334.9 KB

 Archivo verificado y guardado correctamente


---

## ✅ RESUMEN DEL PREPROCESAMIENTO

### 📊 Resultados Obtenidos

**Dataset procesado:**
- **Filas:** 1,000 comentarios
- **Columnas:** 5 (CommentId, VideoId, Text, text_procesado, IsToxic)
- **Sin valores faltantes:** ✅
- **Sin textos vacíos:** ✅

**Reducción de tamaño:**
- Longitud promedio ANTES: 186.0 caracteres
- Longitud promedio DESPUÉS: 112.6 caracteres
- **Reducción: 39.4%** 📉

**Balance del target (IsToxic):**
- Normal (False): 538 comentarios (53.8%)
- Tóxico (True): 462 comentarios (46.2%)
- **Ratio: 1.16x** ✅ (bien balanceado)

---

### 🛠️ Técnicas de Preprocesamiento Aplicadas

1. ✅ **Conversión a minúsculas**
   - Normaliza el texto para que "HELLO" y "hello" sean iguales

2. ✅ **Eliminación de URLs**
   - Remueve enlaces que no aportan al significado

3. ✅ **Eliminación de menciones** (@usuario)
   - Los nombres de usuario no son relevantes para toxicidad

4. ✅ **Eliminación de números**
   - Los números generalmente no aportan al análisis

5. ✅ **Eliminación de caracteres especiales**
   - Solo mantiene letras y espacios

6. ✅ **Eliminación de espacios extra**
   - Limpia espacios múltiples resultantes

7. ✅ **Eliminación de stopwords** (inglés)
   - Remueve 179 palabras comunes (the, and, is, etc.)

8. ✅ **Lematización** (WordNetLemmatizer)
   - Convierte palabras a su raíz lingüística
   - Ejemplo: "running, runs, ran" → "run, run, run"

---

### 📂 Archivos Generados

**Input:**
- `data/raw/youtoxic_original.csv` (original, sin modificar)

**Output:**
- `data/processed/youtoxic_comments_cleaned.csv` ✅

**Estructura del CSV limpio:**
CommentId | VideoId | Text (original) | text_procesado | IsToxic

### 📝 Notas Importantes

- ✅ El preprocesamiento es **determinista** (siempre produce el mismo resultado)
- ✅ Mantenemos `CommentId` y `VideoId` para trazabilidad
- ✅ Mantenemos `Text` original para referencia
- ✅ La columna `text_procesado` es la que usaremos para entrenar

---

**Preprocesamiento completado exitosamente ✅**